In [2]:
import juliapkg
juliapkg.require_julia("=1.12.4")
juliapkg.resolve()

True

In [3]:
from juliacall import Main as jl, convert as jlconvert  # noqa: E402, F401

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [4]:
###############################################################################
# Julia Setup
###############################################################################
def setup_julia():
    """
    Install and import the needed Julia packages.
    """
    print(jl.seval("VERSION"))
    jl.seval("import Pkg")
    jl.seval('Pkg.add("GLM")')
    jl.seval('Pkg.add("MixedModels")')
    jl.seval('Pkg.add("DataFrames")')
    jl.seval('Pkg.add("Distributions")')
    jl.seval('Pkg.add("RCall")')  # for R packages
    jl.seval("using MixedModels")
    jl.seval("using DataFrames")
    jl.seval("using Distributions")
    jl.seval("using GLM")
    jl.seval("using RCall")


    # # Check if 'cocor' R package is installed
    # utils = rpackages.importr('utils')
    # if not rpackages.isinstalled("cocor"):
    #     print("R package 'cocor' not found. Installing now...")
    #     utils.chooseCRANmirror(ind=1) # Select a CRAN mirror
    #     utils.install_packages(StrVector(['cocor']))
    #     print("R package 'cocor' has been installed.")
    # else:
    #     print("R package 'cocor' is already installed.")

    print("Julia environment is set up with MixedModels and DataFrames.")

In [5]:
jl.seval('Pkg.add("StatsBase")')

   Resolving package versions...
    Updating `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Project.toml`
  [2913bbd2] + StatsBase v0.34.10
    Manifest No packages added to or removed from `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Manifest.toml`


In [4]:
setup_julia()

1.12.4


    Updating registry at `C:\Users\deeth\.julia\registries\General.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Project.toml`
    Manifest No packages added to or removed from `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Project.toml`
    Manifest No packages added to or removed from `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Project.toml`
    Manifest No packages added to or removed from `C:\Users\deeth\miniconda3\envs\QA_eyetracking_env\julia_env\Manifest.toml`
   Resolving package versions...
     Project No packages added to or remov

Julia environment is set up with MixedModels and DataFrames.


In [22]:
import numpy as np
import pandas as pd
import scipy.stats
from juliacall import Main as jl, convert as jlconvert  # type: ignore

# -----------------------------
# Julia imports
# -----------------------------
jl.seval("using MixedModels")
jl.seval("using DataFrames")



In [23]:

# -----------------------------
# Simulate data
# -----------------------------
rng = np.random.default_rng(42)

n_subjects = 30
n_texts = 20
n_obs = 2000

subjects = [f"S{i:02d}" for i in range(1, n_subjects + 1)]
texts = [f"T{i:02d}" for i in range(1, n_texts + 1)]

# Observation-level assignments
subject_id = rng.choice(subjects, size=n_obs, replace=True)
text_id = rng.choice(texts, size=n_obs, replace=True)
level = rng.integers(0, 2, size=n_obs)  # binary predictor: 0/1

# Random effects used to generate TF
subject_intercepts = dict(zip(subjects, rng.normal(0, 0.6, size=n_subjects)))
subject_slopes = dict(zip(subjects, rng.normal(0, 0.3, size=n_subjects)))

text_intercepts = dict(zip(texts, rng.normal(0, 0.5, size=n_texts)))
text_slopes = dict(zip(texts, rng.normal(0, 0.2, size=n_texts)))

# Fixed effects
beta_0 = 10.0
beta_level = 2.0

# Generate outcome
TF = []
for s, t, lv in zip(subject_id, text_id, level):
    mu = (
        beta_0
        + beta_level * lv
        + subject_intercepts[s]
        + subject_slopes[s] * lv
        + text_intercepts[t]
        + text_slopes[t] * lv
    )
    y = rng.normal(mu, 1.5)
    TF.append(y)

data = pd.DataFrame({
    "TF": TF,
    "level": level,
    "subject_id": subject_id,
    "text_id": text_id,
})

# Optional: make grouping columns categorical/string-like
data["TF"] = data["TF"].astype(float)
data["level"] = data["level"].astype(int)
data["subject_id"] = data["subject_id"].astype(str)
data["text_id"] = data["text_id"].astype(str)

print(data.head())
print(f"\nSimulated rows: {len(data)}")

          TF  level subject_id text_id
0  10.071859      0        S03     T18
1  13.974021      1        S24     T02
2  11.177998      0        S20     T08
3  12.151760      0        S14     T10
4  11.784484      1        S13     T16

Simulated rows: 2000


In [28]:

# -----------------------------
# Helper for confidence intervals
# -----------------------------
def add_CIs_to_coef_df(coef_df: pd.DataFrame, dof: int):
    coef_df_copy = coef_df.copy()
    t_quantile = scipy.stats.t(df=dof).ppf(0.975)
    coef_df_copy["l_conf"] = (
        coef_df_copy["Coef."] - t_quantile * coef_df_copy["Std. Error"]
    )
    coef_df_copy["u_conf"] = (
        coef_df_copy["Coef."] + t_quantile * coef_df_copy["Std. Error"]
    )
    return coef_df_copy

# -----------------------------
# Mixed model fit in Julia
# -----------------------------
formula = "TF ~ 1 + level + (1+level|subject_id) + (1+level|text_id)"

# jl.seval("global j_df = 0")
# jl.j_df = jlconvert(jl.PyTable, data.dropna())

pan_df = data.dropna().copy()
jl.pan_df = pan_df
jl.seval("using PythonCall")
jl.seval("using DataFrames")
jl.seval("j_df = DataFrame(PythonCall.PyPandasDataFrame(pan_df))")

jl.seval(f"j_formula = @formula({formula})")
jl.seval(
    "mm_model_res = fit(MixedModel, j_formula, j_df, progress=false)"
)

jl.seval(
    """
    function table_to_pd(x)
        PythonCall.Compat.pytable(x)
    end
    """
)

model_res_var_name = "mm_model_res"

mm_coeftable = jl.table_to_pd(
    jl.MixedModels.coeftable(getattr(jl, model_res_var_name))
)
mm_dof = jl.MixedModels.dof(getattr(jl, model_res_var_name))
mm_coeftable = add_CIs_to_coef_df(mm_coeftable, mm_dof)

print("\nCoefficient table with CIs:")
print(mm_coeftable)


Coefficient table with CIs:
          Name      Coef.  Std. Error          z       Pr(>|z|)    l_conf  \
0  (Intercept)  10.069266    0.197934  50.871876   0.000000e+00  9.621508   
1        level   2.066539    0.094290  21.916922  1.791715e-106  1.853241   

      u_conf  
0  10.517023  
1   2.279837  


In [18]:
import pandas as pd
import numpy as np
import pytest

from juliacall import Main as jl, convert as jlconvert


# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------
def try_pytable_conversion(df: pd.DataFrame):
    """
    Try the exact wrapper route that failed for you:
        jlconvert(jl.PyTable, df)

    Returns the converted object if successful, otherwise raises.
    """
    return jlconvert(jl.PyTable, df)


def try_julia_dataframe_from_python_df(df: pd.DataFrame):
    """
    Safer route:
        send pandas DF to Julia
        let Julia build DataFrame(df)
    """
    jl.df_for_test = df
    jl.seval("using DataFrames")
    jl.seval("j_df_for_test = DataFrame(df_for_test)")
    return jl.seval("size(j_df_for_test)")


def try_explicit_column_transfer(df: pd.DataFrame):
    """
    Most robust fallback:
        send plain Python lists column by column.
    """
    for col in df.columns:
        jl.__setattr__(f"col_{col}", df[col].tolist())

    assignments = ",\n".join([f"{col}=col_{col}" for col in df.columns])

    jl.seval("using DataFrames")
    jl.seval(f"j_df_explicit = DataFrame(\n{assignments}\n)")
    return jl.seval("size(j_df_explicit)")


# -------------------------------------------------------------------
# Baseline: plain numeric + string columns
# -------------------------------------------------------------------
def test_pytable_wrapper_works_for_plain_dataframe():
    df = pd.DataFrame({
        "TF": [1.1, 2.2, 3.3],
        "level": [0, 1, 0],
        "subject_id": ["S01", "S02", "S03"],
        "text_id": ["T01", "T02", "T03"],
    })

    wrapped = try_pytable_conversion(df)
    assert wrapped is not None


def test_julia_dataframe_constructor_works_for_plain_dataframe():
    df = pd.DataFrame({
        "TF": [1.1, 2.2, 3.3],
        "level": [0, 1, 0],
        "subject_id": ["S01", "S02", "S03"],
        "text_id": ["T01", "T02", "T03"],
    })

    size = try_julia_dataframe_from_python_df(df)
    assert tuple(size) == (3, 4)


# -------------------------------------------------------------------
# Your actual use case shape
# -------------------------------------------------------------------
def test_pytable_wrapper_with_simulated_mixed_model_style_dataframe():
    rng = np.random.default_rng(42)

    n = 50
    df = pd.DataFrame({
        "TF": rng.normal(size=n).astype(float),
        "level": rng.integers(0, 2, size=n).astype(int),
        "subject_id": rng.choice(["S01", "S02", "S03"], size=n).astype(str),
        "text_id": rng.choice(["T01", "T02", "T03"], size=n).astype(str),
    })

    wrapped = try_pytable_conversion(df)
    assert wrapped is not None


# -------------------------------------------------------------------
# Potential trouble case: pandas categorical
# -------------------------------------------------------------------
def test_pytable_wrapper_may_fail_for_categorical_columns():
    df = pd.DataFrame({
        "TF": [1.1, 2.2, 3.3],
        "level": pd.Series([0, 1, 0], dtype="int64"),
        "subject_id": pd.Series(["S01", "S02", "S03"], dtype="category"),
        "text_id": pd.Series(["T01", "T02", "T03"], dtype="category"),
    })

    # This is intentionally permissive:
    # depending on your installed versions, it may fail or succeed.
    # We mainly want to detect the behavior and document it.
    try:
        wrapped = try_pytable_conversion(df)
        assert wrapped is not None
    except Exception as e:
        assert "PyTable" in str(e) or "convert" in str(e)


def test_safe_route_still_works_after_casting_categorical_to_string():
    df = pd.DataFrame({
        "TF": [1.1, 2.2, 3.3],
        "level": [0, 1, 0],
        "subject_id": pd.Series(["S01", "S02", "S03"], dtype="category").astype(str),
        "text_id": pd.Series(["T01", "T02", "T03"], dtype="category").astype(str),
    })

    size = try_julia_dataframe_from_python_df(df)
    assert tuple(size) == (3, 4)


# -------------------------------------------------------------------
# Potential trouble case: nullable pandas extension dtypes
# -------------------------------------------------------------------
def test_pytable_wrapper_with_nullable_extension_types():
    df = pd.DataFrame({
        "TF": pd.Series([1.1, 2.2, 3.3], dtype="Float64"),   # pandas nullable float
        "level": pd.Series([0, 1, 0], dtype="Int64"),       # pandas nullable int
        "subject_id": ["S01", "S02", "S03"],
        "text_id": ["T01", "T02", "T03"],
    })

    try:
        wrapped = try_pytable_conversion(df)
        assert wrapped is not None
    except Exception as e:
        assert "PyTable" in str(e) or "convert" in str(e)


def test_safe_route_with_normalized_dtypes_after_nullable_columns():
    df = pd.DataFrame({
        "TF": pd.Series([1.1, 2.2, 3.3], dtype="Float64").astype(float),
        "level": pd.Series([0, 1, 0], dtype="Int64").astype(int),
        "subject_id": ["S01", "S02", "S03"],
        "text_id": ["T01", "T02", "T03"],
    })

    size = try_julia_dataframe_from_python_df(df)
    assert tuple(size) == (3, 4)


# -------------------------------------------------------------------
# Missing values
# -------------------------------------------------------------------
def test_pytable_wrapper_with_missing_values():
    df = pd.DataFrame({
        "TF": [1.1, np.nan, 3.3],
        "level": [0, 1, 0],
        "subject_id": ["S01", None, "S03"],
        "text_id": ["T01", "T02", "T03"],
    })

    try:
        wrapped = try_pytable_conversion(df)
        assert wrapped is not None
    except Exception as e:
        assert "PyTable" in str(e) or "convert" in str(e)


def test_safe_route_after_dropna():
    df = pd.DataFrame({
        "TF": [1.1, np.nan, 3.3],
        "level": [0, 1, 0],
        "subject_id": ["S01", None, "S03"],
        "text_id": ["T01", "T02", "T03"],
    }).dropna()

    size = try_julia_dataframe_from_python_df(df)
    assert tuple(size) == (2, 4)


# -------------------------------------------------------------------
# Explicit fallback should basically always work
# -------------------------------------------------------------------
def test_explicit_column_transfer_works():
    df = pd.DataFrame({
        "TF": [1.1, 2.2, 3.3],
        "level": [0, 1, 0],
        "subject_id": ["S01", "S02", "S03"],
        "text_id": ["T01", "T02", "T03"],
    })

    size = try_explicit_column_transfer(df)
    assert tuple(size) == (3, 4)

In [19]:
import pandas as pd
import numpy as np
from juliacall import Main as jl, convert as jlconvert

cases = {
    "plain": pd.DataFrame({
        "TF": [1.1, 2.2, 3.3],
        "level": [0, 1, 0],
        "subject_id": ["S01", "S02", "S03"],
        "text_id": ["T01", "T02", "T03"],
    }),
    "categorical": pd.DataFrame({
        "TF": [1.1, 2.2, 3.3],
        "level": [0, 1, 0],
        "subject_id": pd.Series(["S01", "S02", "S03"], dtype="category"),
        "text_id": pd.Series(["T01", "T02", "T03"], dtype="category"),
    }),
    "nullable": pd.DataFrame({
        "TF": pd.Series([1.1, 2.2, 3.3], dtype="Float64"),
        "level": pd.Series([0, 1, 0], dtype="Int64"),
        "subject_id": ["S01", "S02", "S03"],
        "text_id": ["T01", "T02", "T03"],
    }),
}

for name, df in cases.items():
    print(f"\n--- CASE: {name} ---")
    print(df.dtypes)

    try:
        wrapped = jlconvert(jl.PyTable, df)
        print("PyTable wrapper: SUCCESS")
    except Exception as e:
        print("PyTable wrapper: FAIL")
        print(type(e).__name__, e)

    try:
        jl.df_tmp = df
        jl.seval("using DataFrames")
        jl.seval("j_df_tmp = DataFrame(df_tmp)")
        print("DataFrame(df): SUCCESS")
    except Exception as e:
        print("DataFrame(df): FAIL")
        print(type(e).__name__, e)


--- CASE: plain ---
TF            float64
level           int64
subject_id        str
text_id           str
dtype: object
PyTable wrapper: FAIL
JuliaError cannot convert this Python 'DataFrame' to a Julia 'PyTable'
Stacktrace:
 [1] error(s::String)
   @ Base .\error.jl:44
 [2] macro expansion
   @ C:\Users\deeth\.julia\packages\PythonCall\83z4q\src\Convert\pyconvert.jl:376 [inlined]
 [3] macro expansion
   @ C:\Users\deeth\.julia\packages\PythonCall\83z4q\src\Core\Py.jl:118 [inlined]
 [4] pyconvert(::Type{PyTable}, x::Py)
   @ PythonCall.Convert C:\Users\deeth\.julia\packages\PythonCall\83z4q\src\Convert\pyconvert.jl:390
 [5] (::PythonCall.JlWrap.var"#14#15")(T::Py, x::Py)
   @ PythonCall.JlWrap .\none:1
 [6] pyjlcallback_call(self::PythonCall.JlWrap.var"#14#15", args_::Py, kwargs_::Py)
   @ PythonCall.JlWrap C:\Users\deeth\.julia\packages\PythonCall\83z4q\src\JlWrap\callback.jl:19
 [7] _pyjl_callmethod(f::Any, self_::Ptr{PythonCall.C.PyObject}, args_::Ptr{PythonCall.C.PyObject}, narg

In [20]:
import pandas as pd
from juliacall import Main as jl

df = pd.DataFrame({
    "TF": [1.1, 2.2, 3.3],
    "level": [0, 1, 0],
    "subject_id": ["S01", "S02", "S03"],
    "text_id": ["T01", "T02", "T03"],
})

jl.df = df
jl.seval("using PythonCall")
jl.seval("using DataFrames")

# Explicit pandas wrapper
jl.seval("wrapped = PythonCall.PyPandasDataFrame(df)")
jl.seval("j_df = DataFrame(wrapped)")

print(jl.seval("size(j_df)"))
print(jl.seval("names(j_df)"))

(3, 4)
["TF", "level", "subject_id", "text_id"]
